# 01 — Exploración inicial
## Proyecto final de Machine Learning — Grupo 11
### Dataset: Lending Club Loan Data

**Objetivo de este notebook:** cumplir con la exploración inicial obligatoria de la entrega previa:
carga de datos, tipos y dimensiones, % de faltantes, distribución del target, al menos 5 visualizaciones,
identificación de outliers/registros sospechosos, discusión de sesgos/leakage/limitaciones y un primer baseline simple.

**Nota sobre los datos:** este notebook asume que el archivo `accepted_2007_to_2018Q4.csv.gz`
(o el nombre equivalente del CSV de Lending Club descargado de Kaggle) está ubicado en `../data/raw/`.
El dataset no se distribuye en este repositorio por su tamaño (~1.6 GB); debe descargarse manualmente desde
https://www.kaggle.com/datasets/wordsforthewise/lending-club y colocarse en esa ruta (ver `data/README.md`).


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, roc_auc_score, classification_report,
    confusion_matrix, precision_recall_curve
)

np.random.seed(42)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


ModuleNotFoundError: No module named 'pandas'

## 1. Carga de datos

Dado el tamaño del archivo (~2.2M filas, 140+ columnas), se recomienda:
- Cargar solo un subconjunto de columnas relevantes en una primera pasada, o
- Usar `nrows` / muestreo para la exploración inicial y el pipeline completo para la entrega final.

Aquí se muestra la carga completa de las columnas necesarias para el análisis y el baseline.


In [ ]:
DATA_PATH = "../data/raw/accepted_2007_to_2018Q4.csv"  # ajustar nombre/ruta real del archivo descargado

# Columnas relevantes: identificador, target, fecha de originación y features "leakage-safe"
USECOLS = [
    "id", "loan_status", "issue_d",
    "loan_amnt", "term", "purpose", "annual_inc", "dti", "emp_length",
    "home_ownership", "verification_status", "fico_range_low", "fico_range_high",
    "open_acc", "total_acc", "revol_bal", "revol_util", "delinq_2yrs",
    "earliest_cr_line", "pub_rec", "inq_last_6mths", "mort_acc",
    "application_type", "addr_state",
    # variables "zona gris" (se analizan aparte, no se incluyen en el baseline)
    "grade", "sub_grade", "int_rate",
]

df = pd.read_csv(DATA_PATH, usecols=USECOLS, low_memory=False)
df.shape


## 2. Definición de la variable objetivo

Se colapsa `loan_status` en una variable binaria y se excluyen los préstamos cuyo desenlace
aún no está definido (`Current`, `In Grace Period`, `Late (16-30 days)`, `Late (31-120 days)`),
para evitar sesgo de censura (ver `proposal.md`, sección 5 y 8).


In [ ]:
print("Distribución original de loan_status:")
print(df["loan_status"].value_counts(dropna=False))

MALOS = ["Charged Off", "Default"]
BUENOS = ["Fully Paid"]

df_model = df[df["loan_status"].isin(MALOS + BUENOS)].copy()
df_model["target"] = df_model["loan_status"].isin(MALOS).astype(int)

print(f"\nFilas excluidas por estado no definitivo: {len(df) - len(df_model):,}")
print(f"Filas para modelado: {len(df_model):,}")


## 3. Número de filas, columnas y tipos de datos

In [ ]:
print(f"Shape (post-filtro de target): {df_model.shape}")
df_model.dtypes.value_counts()


In [ ]:
df_model.dtypes.sort_values().to_frame(name='dtype')

## 4. Porcentaje de valores faltantes por variable

In [ ]:
missing_pct = (df_model.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
missing_pct.to_frame(name="% faltante")


In [ ]:
# Visualización 1: % de faltantes por variable
fig, ax = plt.subplots(figsize=(8, 5))
missing_pct.plot(kind="barh", ax=ax, color="indianred")
ax.set_xlabel("% de valores faltantes")
ax.set_title("Porcentaje de valores faltantes por variable")
plt.tight_layout()
plt.show()


## 5. Distribución de la variable objetivo

In [ ]:
target_counts = df_model["target"].value_counts(normalize=True) * 100
print(target_counts)

# Visualización 2: distribución del target
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="target", data=df_model, ax=ax, palette=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["Bueno (Fully Paid)", "Malo (Default/Charged Off)"])
ax.set_title("Distribución de la variable objetivo")
ax.set_xlabel("")
plt.tight_layout()
plt.show()


**Lectura esperada:** se anticipa un fuerte desbalance de clases (~15-20% de préstamos en default/charged off),
lo cual justifica usar AUC-PR y KS como métricas principales en vez de accuracy (ver `proposal.md`, sección 9).


## 6. Visualizaciones adicionales

In [ ]:
# Visualización 3: distribución de loan_amnt por clase
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df_model, x="loan_amnt", hue="target", bins=50, kde=True, ax=ax,
             palette=["#4C72B0", "#C44E52"], stat="density", common_norm=False)
ax.set_title("Distribución del monto del préstamo por clase")
plt.tight_layout()
plt.show()


In [ ]:
# Visualización 4: dti (debt-to-income) por clase (boxplot, con recorte de outliers extremos)
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x="target", y="dti", data=df_model[df_model["dti"].between(0, 60)], ax=ax)
ax.set_xticklabels(["Bueno", "Malo"])
ax.set_title("DTI (debt-to-income) por clase")
plt.tight_layout()
plt.show()


In [ ]:
# Visualización 5: tasa de default por rango de FICO
df_model["fico_avg"] = (df_model["fico_range_low"] + df_model["fico_range_high"]) / 2
df_model["fico_bin"] = pd.cut(df_model["fico_avg"], bins=10)

fico_default_rate = df_model.groupby("fico_bin", observed=True)["target"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
fico_default_rate.plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("Tasa de default")
ax.set_title("Tasa de default por rango de FICO score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Visualización 6: tasa de default por propósito del préstamo
purpose_default = df_model.groupby("purpose")["target"].agg(["mean", "count"]).sort_values("mean", ascending=False)
purpose_default = purpose_default[purpose_default["count"] > 1000]  # descartar categorías con muy pocos casos

fig, ax = plt.subplots(figsize=(8, 5))
purpose_default["mean"].plot(kind="barh", ax=ax, color="darkorange")
ax.set_xlabel("Tasa de default")
ax.set_title("Tasa de default por propósito del préstamo (categorías con >1000 casos)")
plt.tight_layout()
plt.show()


In [ ]:
# Visualización 7: matriz de correlación entre variables numéricas
num_cols = ["loan_amnt", "annual_inc", "dti", "fico_avg", "open_acc",
            "total_acc", "revol_bal", "revol_util", "delinq_2yrs",
            "inq_last_6mths", "int_rate", "target"]

corr = df_model[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriz de correlación (variables numéricas)")
plt.tight_layout()
plt.show()


## 7. Identificación de outliers y registros sospechosos

Se revisan valores extremos o inconsistentes en variables clave.


In [ ]:
print("annual_inc — estadísticos:")
print(df_model["annual_inc"].describe())
print(f"\nPréstamos con annual_inc > $1,000,000: {(df_model['annual_inc'] > 1_000_000).sum():,}")

print("\ndti — estadísticos:")
print(df_model["dti"].describe())
print(f"\nPréstamos con dti negativo (inconsistente): {(df_model['dti'] < 0).sum():,}")
print(f"Préstamos con dti > 100 (sospechoso): {(df_model['dti'] > 100).sum():,}")

print(f"\nPréstamos con revol_util > 150% (posible error de captura): {(df_model['revol_util'] > 150).sum():,}")


**Decisión preliminar:** los registros con `annual_inc` extremo (>$1M) o `dti` fuera de rango razonable
(negativo o >100) se tratarán como candidatos a exclusión o a winsorización en la etapa de limpieza
(`02_limpieza_features.ipynb`), documentando el criterio y el número de filas afectadas.


## 8. Discusión inicial de sesgos, leakage y limitaciones

**Leakage:**
- Se excluyeron explícitamente todas las variables generadas después del desembolso del préstamo
  (`total_pymnt`, `recoveries`, `last_pymnt_d`, `hardship_*`, `settlement_*`, etc.), ya que revelan directamente el desenlace.
- `grade`, `sub_grade` e `int_rate` se mantienen como "zona gris": están disponibles al momento de originación,
  pero son producto del propio modelo de scoring de Lending Club. Se entrenará con y sin estas variables
  para medir cuánto del desempeño depende de "copiar" la decisión de LC.

**Sesgos:**
- El dataset solo contiene préstamos **aprobados** por Lending Club (no hay información de solicitudes rechazadas),
  por lo que cualquier modelo entrenado aquí hereda el sesgo de selección del proceso de aprobación original.
- Variables como `addr_state` o `home_ownership` pueden actuar como proxies de variables socioeconómicas
  sensibles; se documentará si el modelo genera disparidades relevantes por segmento en el análisis de errores.
- El comportamiento de default varía por ciclo económico (ej. prestamos originados antes/durante la crisis
  de 2008-2009 vs. después); esto motiva el split temporal (out-of-time) descrito en `proposal.md`.

**Limitaciones:**
- No se dispone de las solicitudes rechazadas, por lo que no se puede modelar el proceso de aprobación completo,
  solo el riesgo condicional a haber sido aprobado.
- Variables de texto libre (`emp_title`, `desc`) no se usarán en el baseline por su alta cardinalidad y ruido;
  se evaluará su potencial en iteraciones posteriores.


## 9. Baseline simple

Regresión logística con las variables "incluidas" (sin `grade`/`sub_grade`/`int_rate`), split **temporal**
(no aleatorio) usando `issue_d`, siguiendo el plan de validación de `proposal.md`.


In [ ]:
df_model["issue_d"] = pd.to_datetime(df_model["issue_d"], format="%b-%Y")
df_model["issue_year"] = df_model["issue_d"].dt.year

train = df_model[df_model["issue_year"].between(2015, 2017)]
valid = df_model[df_model["issue_year"] == 2018]

print(f"Train: {len(train):,} filas | Valid: {len(valid):,} filas")
print(f"Tasa de default train: {train['target'].mean():.3f} | valid: {valid['target'].mean():.3f}")


In [ ]:
FEATURES_NUM = ["loan_amnt", "annual_inc", "dti", "fico_avg", "open_acc",
                "total_acc", "revol_bal", "revol_util", "delinq_2yrs",
                "inq_last_6mths", "mort_acc"]
FEATURES_CAT = ["term", "purpose", "emp_length", "home_ownership",
                "verification_status", "application_type"]

X_train, y_train = train[FEATURES_NUM + FEATURES_CAT], train["target"]
X_valid, y_valid = valid[FEATURES_NUM + FEATURES_CAT], valid["target"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), FEATURES_NUM),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), FEATURES_CAT),
])

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

baseline.fit(X_train, y_train)


In [ ]:
y_pred_proba = baseline.predict_proba(X_valid)[:, 1]
y_pred = baseline.predict(X_valid)

print(f"AUC-ROC: {roc_auc_score(y_valid, y_pred_proba):.3f}")
print(f"AUC-PR (métrica principal): {average_precision_score(y_valid, y_pred_proba):.3f}")
print()
print(classification_report(y_valid, y_pred, target_names=["Bueno", "Malo"]))
print()
print("Matriz de confusión:")
print(confusion_matrix(y_valid, y_pred))


**Lectura inicial:** este resultado es el piso de comparación honesto para el proyecto.
En las siguientes iteraciones se compararán modelos de árboles y gradient boosting contra este baseline,
y se evaluará el efecto de incluir las variables "zona gris" (`grade`, `sub_grade`, `int_rate`).
